# Classical Experiment 7 — Specific Heats: the Third Thermodynamic Crisis *(supplement)*
**Not covered by Weinberg ch. 1** — included to complete the thermodynamics thread of
`exp1_blackbody_thermodynamics`: the other great equipartition failure of 1900–1912.
**What the experiments showed (and where classical physics broke)**

- **Dulong–Petit (1819):** molar heat of solids $C_V \approx 3R \approx 5.96$ cal/mol·K — explained classically
  (6 quadratic degrees of freedom, equipartition $k_{\rm B}T/2$ each). For a *century* this looked like a law.
- **Observed (Richards 1897, Dewar 1905, Nernst & Lindemann 1911, Eucken 1912):** $C_V \to 0$ as $T \to 0$
  (Nernst's heat theorem context); diamond at room temperature sits at barely half of Dulong–Petit.
- **Einstein (1907):** $N$ identical oscillators of frequency $\nu_E$ with Planck quantization,
  $$C_V = 3R\left(\frac{\theta_E}{T}\right)^2 \frac{e^{\theta_E/T}}{(e^{\theta_E/T}-1)^2}, \qquad \theta_E = \frac{h\nu_E}{k_{\rm B}},$$
  predicts freeze-out — but exponentially fast, whereas experiments show a **power law**.
- **Debye (1912):** the solid supports a *continuum* of elastic modes up to $\nu_{\max}$,
  $$C_V = 9R\left(\frac{T}{\theta_D}\right)^3 \int_0^{\theta_D/T}\!\! dx\, x^4 \frac{e^x}{(e^x-1)^2}
  \;\xrightarrow[T\ll\theta_D]{}\; \frac{12\pi^4}{5}R\left(\frac{T}{\theta_D}\right)^3,$$
  matching the observed $T^3$ law. Yet another place where replacing continuous energy with $h\nu$-quantized
  packets is the *only* fix.
- **Metals (bonus, later):** conduction electrons should add $\tfrac32 R$ (classical free gas) — they add almost
  nothing; $C_V = \gamma T + \beta T^3$ at low $T$ is the lattice $T^3$ plus a linear electronic tail
  (Fermi–Dirac, ch. 4/5 territory). Two power laws on one log–log plot.


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from scipy.integrate import quad
R = 8.314462618                     # J/mol K
Rcal = R/4.184

def C_einstein(T, ThE):
    x = np.minimum(ThE/np.asarray(T, dtype=float), 300.0)   # clamp: function is 0 far below theta_E anyway
    with np.errstate(over='ignore'):
        return 3*Rcal*(x/(2*np.sinh(x/2)))**2    # stable form of x^2 e^x/(e^x-1)^2

def C_debye(T, ThD):
    T = np.asarray(T, dtype=float)
    def integr(x):
        if x > 1e-8:
            with np.errstate(over='ignore'):
                return x**4/(4*np.sinh(x/2)**2)      # stable form of x^4 e^x/(e^x-1)^2
        return x**2
    up = ThD/T
    return np.array([9*Rcal*(T_/ThD)**3 * quad(integr, 0, u, epsrel=1e-10)[0] for T_, u in zip(np.atleast_1d(T), np.atleast_1d(up))]).ravel()

print(f"Dulong-Petit value 3R = {3*Rcal:.3f} cal/mol K")
print(f"Debye T^3 coefficient check: 12 pi^4 R/5 = {12*np.pi**4*Rcal/5:.3f} cal/mol K x (T/theta_D)^3")


## 1. Dulong–Petit as the classical line, and its failures
Room-temperature $C_V$ for common solids against the classical $3R$ line. Diamond and silicon fall far below —
classical equipartition has no temperature scale, so this is a flat-out contradiction.

In [ ]:
solids = [('Pb', 6.4, 88), ('Cu', 5.85, 315), ('Al', 5.82, 428), ('Si', 4.7, 645), ('C (diamond)', 3.0, 2230)]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.axhline(3*Rcal, color='tab:red', ls='--', lw=2, label=r'classical Dulong-Petit $3R$')
xs = np.arange(len(solids))
ax.bar(xs, [s[1] for s in solids], color=['#79c','#79c','#79c','#e97','#e97'], width=0.55,
       label=r'observed $C_V$ at room $T$')
for x, (nm, cv, th) in zip(xs, solids):
    ax.text(x, cv+0.08, f'{nm}\n$\\theta_D$={th} K', ha='center', fontsize=8)
ax.set_xticks([]); ax.set_ylabel(r'$C_V$ (cal mol$^{-1}$ K$^{-1}$)')
ax.set_title('Dulong-Petit: a century-old law, systematically violated')
ax.legend(fontsize=9)
plt.show()


## 2. Einstein vs Debye: the shape of freeze-out
Einstein: exponential freeze-out below $\theta_E$. Debye: a whole band of modes $\nu \le \nu_{\max}$ — soft modes
survive to lower $T$, giving the observed $T^3$. Curves for several characteristic temperatures (Pb-like to
diamond-like), log temperature axis.

In [ ]:
T = np.logspace(0.3, np.log10(1500), 300)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
for Th, cc in [(88, '#79c'), (315, '#5a5'), (645, '#e97'), (2230, '#c33')]:
    axes[0].plot(T, C_einstein(T, 1.35*Th)/3/Rcal, lw=2, color=cc, label=f'$\\theta_E$={1.35*Th:.0f} K')
    axes[1].plot(T, C_debye(T, Th)/3/Rcal, lw=2, color=cc, label=f'$\\theta_D$={Th} K')
    axes[1].plot(T, 4*np.pi**4/5*(T/Th)**3/3, ':', lw=1, color=cc)
for ax_ in axes:
    ax_.set_xscale('log'); ax_.set_ylim(-0.02, 1.05)
    ax_.axhline(1, color='k', lw=0.6, ls='--')
    ax_.set_xlabel('$T$ (K)'); ax_.set_ylabel(r'$C_V/3R$')
axes[0].set_title('(a) Einstein (1907): exponential freeze-out'); axes[0].legend(fontsize=8)
axes[1].set_title('(b) Debye (1912): $T^3$ at low $T$ (dotted)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 3. The law of corresponding states: all solids on one curve
Plotting $C_V/3R$ against $T/\theta_D$ collapses every solid onto the single Debye curve — the graphical signature
that one quantum scale $h\nu_{\max}/k_{\rm B}$ controls the whole phenomenon (and the pattern black-body radiation
set: Wien's $\nu/T$ scaling, exp. 1).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
Tt = np.logspace(-2.3, 0.7, 200)
ax.plot(Tt, C_debye(2230*Tt, 2230)/3/Rcal, lw=2.5, color='k', label='universal Debye curve')
for nm, cv_rt, Th in solids:
    Tt_rt = 298/Th
    ax.plot([Tt_rt], [cv_rt/3/Rcal], 'o', ms=8, label=nm)
ax.plot(Tt, 4*np.pi**4/15*Tt**3, '--', lw=1.5, color='tab:red', label=r'$T^3$ asymptote $\frac{4\pi^4}{15}(T/\theta_D)^3$')
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_ylim(1e-3, 1.4)
ax.set_xlabel(r'$T/\theta_D$'); ax.set_ylabel(r'$C_V/3R$')
ax.set_title('Corresponding states: one quantum scale per solid'); ax.legend(fontsize=8)
plt.show()


## 4. Metals: two power laws at once
Classical free-electron gas would add $\tfrac32 R$, destroying Dulong–Petit outright. Observed low-$T$ law:
$C_V = \gamma T + \beta T^3$: log–log shows the crossover between the linear electronic tail and the lattice $T^3$.

In [ ]:
gamma, beta = 0.688, 0.0479            # Cu, mJ/mol K^2, mJ/mol K^4 (order of magnitude)
T = np.logspace(0.3, 2.3, 200)
C = gamma*T + beta*T**3
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(T, C, lw=2.5, color='k', label=r'$C_V=\gamma T+\beta T^3$ (Cu)')
ax.loglog(T, gamma*T, '--', lw=1.2, label=r'electronic $\gamma T$')
ax.loglog(T, beta*T**3, '--', lw=1.2, label=r'lattice $\beta T^3$')
ax.loglog(T, 3*Rcal*1e3*np.ones_like(T), ':', lw=1.5, color='tab:red', label=r'classical $3R$ (+$\frac{3}{2}R$ e$^-$)')
ax.set_xlabel('$T$ (K)'); ax.set_ylabel(r'$C_V$ (mJ mol$^{-1}$ K$^{-1}$)')
ax.set_title('Specific heat of copper at low $T$'); ax.legend(fontsize=8)
plt.show()


## My summary
- Dulong–Petit is equipartition's last stand in thermodynamics: $3R$ independent of $T$ and of the material.
  Both fail — at low $T$ everything freezes out, and the freeze-out temperature varies wildly by material.
- Einstein's single-frequency model captures freeze-out but gets the *form* wrong (exponential, not $T^3$);
  Debye's continuum of modes fixes it, giving the law of corresponding states $C_V = 3R\,f(T/\theta_D)$.
- Same pattern as black-body radiation: classical continuous energies give a wrong *temperature-independent or
  divergent* answer; Planck quantization with one new scale ($h$) fixes both.
- Metals hide a second crisis: conduction electrons contribute $\sim\gamma T$, not $\tfrac32 R$ — the free-electron
  gas is also quantized (Fermi statistics, later chapters).

## Gaps
- Debye treats the solid as an elastic continuum with a sharp cutoff — where does $\nu_{\max}$ come from
  (one mode per atom, $3N$ modes total)? Reproduce Eq. above symbolically.
- Why does Einstein's single-mode picture work better for *optical* modes in crystals with two atoms per cell?
- Connect Nernst's heat theorem ($C_V \to 0$) to the third law — outside Weinberg, flag for stat-mech review.